In [23]:
import pandas as pd

# Load the full prices data
full_prices = pd.read_csv("full_gas_prices.csv")


In [24]:
# Rename the column
full_prices.rename(columns={'Unnamed: 0': 'Date'}, inplace=True)

# Convert to datetime
full_prices['Date'] = pd.to_datetime(full_prices['Date'])

# Set as index
full_prices.set_index('Date', inplace=True)

# Check result
full_prices.head()

,Prices
Date,
2020-10-31,10.100000
2020-11-01,10.106667
2020-11-02,10.113333
2020-11-03,10.120000
2020-11-04,10.126667


In [25]:
# Simple price lookup function
def get_price(date):
    date = pd.to_datetime(date)
    return full_prices.loc[date, 'Prices']


def price_contract(injection_date, withdrawal_date, volume,
                   storage_cost_per_day,
                   injection_cost_per_unit,
                   withdrawal_cost_per_unit,
                   max_storage,
                   injection_rate,
                   withdrawal_rate):

    total_profit = 0
    current_storage = 0
    
    for injection_date, withdrawal_date in zip(injection_date, withdrawal_date):
        
        # Convert dates
        injection_date = pd.to_datetime(injection_date)
        withdrawal_date = pd.to_datetime(withdrawal_date)

        # Calculate how many days needed
        injection_days = volume / injection_rate
        withdrawal_days = volume / withdrawal_rate

        # Effective storage period
        effective_start = injection_date + pd.Timedelta(days=injection_days)
        effective_end = withdrawal_date

        # Check storage capacity before injection
        if current_storage + volume > max_storage:
            print("Exceeded storage capacity")
            return None
            
        # Get prices
        buy_price = get_price(injection_date)
        sell_price = get_price(withdrawal_date)

        # Add gas to storage
        current_storage += volume
        
        # Calculate storage time
        days_stored = (withdrawal_date - injection_date).days
        
        # Costs
        storage_cost = days_stored * storage_cost_per_day
        injection_cost = volume * injection_cost_per_unit

        # Remove gas at withdrawal
        if current_storage < volume:
            print("Not enough gas to withdraw")
            return None
        
        current_storage -= volume
        
        withdrawal_cost = volume * withdrawal_cost_per_unit
    
        # Sum of all costs
        all_costs = storage_cost+injection_cost+withdrawal_cost
        
        # Final profit
        profit = (sell_price - buy_price) * volume - all_costs
        total_profit += profit
    
    return total_profit

In [27]:
price_contract(
    injection_date=["2023-05-01", "2023-06-01"],
    withdrawal_date=["2023-11-01", "2023-12-01"],
    volume=500000,
    storage_cost_per_day=1000,
    injection_cost_per_unit=0.01,
    withdrawal_cost_per_unit=0.01,
    max_storage=1000000
)


np.float64(289182.7956989261)